In [18]:
!pip install face_recognition

import cv2
import os
import numpy as np
import face_recognition
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

dataset_path = "/content/drive/MyDrive/FaceProject/oyuncu_yuzleri"

test_path = "/content/drive/MyDrive/FaceProject/test_images"

known_face_encodings = []
known_face_names = []

print("Kayıtlı yüzler yükleniyor...\n")

for person_name in os.listdir(dataset_path):

    person_folder = os.path.join(dataset_path, person_name)

    if os.path.isdir(person_folder):

        for image_name in os.listdir(person_folder):

            image_path = os.path.join(person_folder, image_name)

            try:
                image = face_recognition.load_image_file(image_path)

                encodings = face_recognition.face_encodings(image)

                if len(encodings) > 0:

                    known_face_encodings.append(encodings[0])
                    known_face_names.append(person_name)

                    print(f"Yüklendi -> {person_name} / {image_name}")

            except:
                print(f"Hata oluştu -> {image_name}")

print("\nTüm kayıtlı yüzler yüklendi.\n")

test_images = os.listdir(test_path)

for test_image_name in test_images:

    test_image_path = os.path.join(test_path, test_image_name)

    print(f"\nTest ediliyor -> {test_image_name}")

    image = cv2.imread(test_image_path)

    if image is None:
        print("Görsel okunamadı.")
        continue

    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(50, 50)
    )

    print(f"Bulunan yüz sayısı: {len(faces)}")

    for (x, y, w, h) in faces:
        face_location = (y, x + w, y + h, x)

        encodings = face_recognition.face_encodings(rgb_image, [face_location])

        name = "Kayitli Degil"
        similarity_score = 0

        if len(encodings) > 0:

            face_encoding = encodings[0]

            matches = face_recognition.compare_faces(
                known_face_encodings,
                face_encoding,
                tolerance=0.5
            )

            face_distances = face_recognition.face_distance(
                known_face_encodings,
                face_encoding
            )

            best_match_index = np.argmin(face_distances)
            similarity_score = (1 - face_distances[best_match_index]) * 100

            if matches[best_match_index]:
                name = known_face_names[best_match_index]

        print(f"Benzerlik Oranı: %{similarity_score:.2f} ({name})")

        color = (0, 0, 255) if name == "Kayitli Degil" else (0, 255, 0)

        cv2.rectangle(
            image,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        display_text = f"{name} (%{similarity_score:.1f})"
        cv2.putText(
            image,
            display_text,
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

    plt.figure(figsize=(8,8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(test_image_name)
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.